# IBKR capture and the entropy gate (runs locally)

Records quotes, trades and order-book depth from TWS, then turns the capture into the same 5-minute bars the rest of the project uses, so the gate can run on it. **Market data only: nothing here places an order.**

**Before you run it**
- TWS is open and logged into the **paper** account, with the API enabled on port 7497 (*Global Configuration → API → Settings*).
- It is a weekday between **09:30 and 16:00 ET**. The first capture ran at 19:54 ET, after the close, which is why depth came back empty and AAPL showed a 9 bps spread (regular hours are closer to 0.3).
- Keep the Mac awake while recording: run `caffeinate -dims` in a Terminal window.
- Depth is a separate paid subscription per exchange. If a stream fails, the message printed below says why.

Rows are saved to `data/ibkr/` every 5 seconds by a background thread, so a crashed kernel loses seconds, not a session.

In [ ]:
import sys, threading, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.ibkr_capture import CaptureApp, market_open_now

HOST, PORT, CLIENT_ID = "127.0.0.1", 7497, 7
OUTPUT_DIR = ROOT / "data" / "ibkr"

# IBKR allows only a few depth streams at once (3 by default), so start small.
SYMBOLS = ["SPY", "XLK"]
WITH_DEPTH = True

print("market open right now:", market_open_now())

## 1. Connect

In [ ]:
app = CaptureApp(OUTPUT_DIR)
app.connect(HOST, PORT, CLIENT_ID)
threading.Thread(target=app.run, daemon=True).start()

deadline = time.time() + 10
while not app.connected and time.time() < deadline:
    time.sleep(0.1)
print("Ready." if app.connected else f"Could not connect. Is TWS running with the API enabled on port {PORT}?")

## 2. Start recording

Depth is requested as SMART depth. If it fails with 354 or 10089, the code is fine and the account lacks a depth subscription; add one in Client Portal under *Settings → Market Data Subscriptions*, and for a paper account turn on sharing of live subscriptions. To ask one exchange directly instead, use `app.subscribe("SPY", depth_exchange="ARCA")`.

In [ ]:
for symbol in SYMBOLS:
    app.subscribe(symbol, depth=WITH_DEPTH)
app.start_autoflush(interval=5)
print("recording:", SYMBOLS)

## 3. Check on it

Run this any time. `quote_age_s` should stay in single digits during regular hours; if it keeps growing the stream has stalled.

In [ ]:
display(pd.DataFrame(app.status()))
for symbol in SYMBOLS:
    book = app.get_current_book(symbol)
    print(f"\n{symbol}  best 3 levels")
    for side in ("bid", "ask"):
        print(f"  {side}:", [(p, s) for p, s, _ in book[side][:3]] or "empty")
print("\nerrors so far:", [(code, msg[:60]) for _, code, msg in app.errors] or "none")

## 4. Stop

Run when you are done for the session.

In [ ]:
app.unsubscribe_all()
app.stop_autoflush()
app.disconnect()
print("Stopped. Files:", sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))

---
## Running the idea on what was captured

Reads every saved day, builds regular-hours 5-minute bars from the **quote midpoint**, and runs the gate on them. Midpoint bars do not bounce between bid and ask the way trade prices do, so they are a cleaner price series than the Alpaca trade bars but not the same one. Compare the two feeds on a shared day before letting either stand in for the other.

The gate reads 5 sessions of returns and scores the one after, so **the first reading needs 6 complete sessions**. A session still in progress is not counted.

In [ ]:
from src.config import load_config
from src.ibkr import capture_grids, read_ticks, ticks_to_bars
from src.screening import apply_gate, screen_universe

files = sorted(OUTPUT_DIR.glob("ticks_*.csv"))
ticks = pd.concat([read_ticks(f) for f in files], ignore_index=True) if files else pd.DataFrame()
print(f"{len(files)} day file(s), {len(ticks):,} tick rows")

grids = capture_grids(ticks) if len(ticks) else {}
rows = []
for symbol, g in grids.items():
    real = g["close"].notna()
    rows.append({
        "symbol": symbol, "sessions": g["session_id"].nunique(),
        "bars with a quote": f"{real.mean():.1%}",
        "median spread (bps)": round(g["spread_bps"].median(), 2),
        "quotes per bar": round(g["quotes"].mean(), 1),
        "trades per bar": round(g["trades"].mean(), 1),
    })
display(pd.DataFrame(rows) if rows else "No complete regular-hours session captured yet.")

sessions = max((g["session_id"].nunique() for g in grids.values()), default=0)
print(f"\ncomplete sessions: {sessions}   sessions still needed for the first gate reading: {max(0, 6 - sessions)}")

In [ ]:
cfg = load_config(ROOT / "configs/etf_intraday.yaml")
table = screen_universe(grids, cfg.screening) if grids else pd.DataFrame()
if table.empty:
    print("No gate readings yet.")
else:
    s = cfg.screening
    gated = apply_gate(table, s.gate_statistic, s.alpha, s.max_tie_fraction, s.min_ticks_per_bar)
    cols = ["symbol", "session_date", "trade_date", "pe_unweighted", "p_trend", "ticks_per_bar", "tick_limited", "passed"]
    display(gated[cols].tail(15))

## The depth file

Checks that depth is arriving and that the saved messages can rebuild the book. `operation` is 0 for insert, 1 for update, 2 for delete; the first recorder did not save it.

In [ ]:
from src.ibkr import replay_book

depth_files = sorted(OUTPUT_DIR.glob("depth_*.csv"))
if not depth_files:
    print("No depth file yet. Depth was not requested, or every request failed (see the errors printed above).")
else:
    depth = pd.concat([pd.read_csv(f) for f in depth_files], ignore_index=True)
    print(f"{len(depth):,} depth messages")
    display(depth.groupby(["symbol", "operation"]).size().unstack(fill_value=0).rename(columns={0: "insert", 1: "update", 2: "delete"}))
    last_day = pd.read_csv(depth_files[-1])
    for symbol in last_day["symbol"].unique():
        book = replay_book(last_day, symbol, max_rows=10)
        print(f"\n{symbol} book after the last message (from the file alone)")
        for side in ("bid", "ask"):
            print(f"  {side}:", [(p, s) for p, s, _ in book[side][:3]])

## A look at the first capture (after hours)

The recording from 19:54 to 19:58 ET on the first version of the recorder. It has no regular-hours bars, so this shows the extended-hours bars instead.

In [ ]:
first = Path.home() / "Downloads/ibkr_microstructure_data/tick_by_tick.csv"
if first.exists():
    old = read_ticks(first)
    print("regular-hours bars:", len(ticks_to_bars(old)))
    display(ticks_to_bars(old, regular_hours_only=False).round(3))
else:
    print("first capture not found at", first)